# [실습] 분류 모델 지표와 데이터 불균형

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🎯 [실습] 분류 지표와 불균형 — 어떤 실수를 감수할지 고른다

## — 0.89라는 숫자 뒤에, 놓친 사람이 몇 명인지 센다

지난 두 순서 동안 정확도만 봤습니다. 0.8451에서 0.8973까지 올렸고, 그 숫자가 흔들리지 않는다는 것도 확인했습니다. 그런데 한 번도 묻지 않은 질문이 있습니다 — **정확히 무엇을 맞히고 무엇을 틀렸나?**

> 🤖 **오늘의 AI 활용 규칙 — 검증 단계:**  
> 코드를 AI에게 물어도 됩니다. 단, **지표 이름을 정확히 지정**해서 묻고, 돌아온 숫자가 혼동행렬과 맞는지 직접 대조합니다. 지표를 잘못 고르는 것은 AI가 대신 책임져 주지 않습니다. (자세한 규칙은 개념 노트북 Part 0)

## 📋 오늘의 미션

마케팅팀이 구체적인 계획을 들고 왔습니다.

> 🧑‍💼 "구매로 이어질 것 같은 세션에 **무료배송 쿠폰**을 띄우려 합니다. 다음 달 예산으로 **500세션**까지 가능합니다. 어떤 세션을 골라야 할지, 그리고 그렇게 하면 **구매 건수를 몇 건이나 챙길 수 있는지** 알려주세요."

이 요청에는 두 가지가 들어 있습니다. **500이라는 용량 제약**과 **성과의 근거**입니다. 정확도로는 어느 쪽도 답할 수 없습니다. 오늘 그 답을 만듭니다.

| 문제 | 내용 | 확인하는 힘 |
| --- | --- | --- |
| 1 | 혼동행렬로 0.89를 쪼갠다 | 정확도의 착시 확인 |
| 2 | 지표 5종을 교차 검증으로 | 문제에 맞는 지표 고르기 |
| 3 | 임계값을 훑어 운영점 선택 | 용량 제약을 숫자로 번역 |
| 4 | `class_weight` 전후 비교 | 보정 채택 여부 결정 |
| 5 | 모델 카드 v4 완성 | 지표 선택 근거 기록 (**제출물**) |

> ⚠️ **미리 밝혀 둘 한계:**  
> 엄밀히 말하면 골라야 할 대상은 *"쿠폰이 있어야만 사는 세션"* 입니다. 그런데 우리 모델이 예측하는 것은 *"살 가능성이 높은 세션"* 이라 둘이 완전히 같지는 않습니다(개입의 효과를 예측하는 것은 더 어려운 문제입니다). 오늘은 후자로 진행하되, **이 차이를 모델 카드의 한계 항목에 적는 것**까지가 과제입니다.

# ⚙️ 데이터 준비

같은 데이터를 이어서 씁니다. 모델도 지난 순서에서 **채택한 것**을 동일하게 사용합니다 — 랜덤포레스트에 `min_samples_leaf=20`을 준 설정입니다(CV 정확도 0.8973, 학습-테스트 격차 0.0263).

오늘 바뀌는 것은 모델이 아니라 **모델을 읽는 방법**입니다.

| 데이터 | 내용 | 출처·라이선스 |
| --- | --- | --- |
| **Online Shoppers Purchasing Intention** | 온라인 스토어 방문 세션 12,330건 — 18개 열, 타깃 `Revenue`(구매 여부, 양성 15.5%) | [UCI 468](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset) · CC BY 4.0 |

▶️ **코드 실행하기 · 코드 셀 1 [C1]**

In [1]:
# ─────────────────────────────────────────────
# [C1] ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

X = shoppers[NUM_COLS]
y = shoppers["Revenue"].astype(int)

print(f"쇼핑 세션 데이터: {shoppers.shape[0]:,}행 × {shoppers.shape[1]}열")
print(f"구매 전환율(양성 비율): {y.mean():.4f}")
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
쇼핑 세션 데이터: 12,330행 × 18열
구매 전환율(양성 비율): 0.1547

→ 준비 완료. 이제 여러분 차례입니다.


# 문제 1. 혼동행렬로 0.89를 쪼갠다

정확도 0.8973은 "2,466건 중 약 2,213건을 맞혔다"는 뜻입니다. 그런데 **맞힌 것의 대부분이 '안 산다'** 였다면 이 정확도만으로 마케팅팀의 구매 세션 포착 목적을 충족했는지 판단할 수 없습니다. 네 칸으로 쪼개서 확인합니다.

```
[문제 1]
1) 지난 순서에서 채택한 모델을 학습합니다.
   RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
2) 테스트셋에 대한 혼동행렬을 구해 TN · FP · FN · TP를 각각 출력합니다.
3) 정확도 · 정밀도 · 재현율 · F1을 계산해 함께 출력합니다.
4) "실제 구매 세션 중 몇 건을 놓쳤는가"를 건수와 비율로 설명합니다.
```

> 🤔 **예상하기**  
> 테스트셋 2,466건 중 실제 구매는 **382건**입니다. 정확도 0.89인 이 모델이 그 382건 중 몇 건이나 찾아낼 것 같습니까? 300건, 200건, 100건 가운데 예상값을 고릅니다.

▶️ **코드 실행하기 · 코드 셀 2 [C2]**

In [2]:
# [C2] 문제 1. 혼동행렬로 0.89를 쪼갠다
# ⌨️ 문제 1 — 정확도 하나를 네 칸으로 쪼개기
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score)

# 여기에 코드를 작성하세요 (분리 → 학습 → 혼동행렬 → 지표 4종 → 놓친 사람 수)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 1) 분리
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2) 모델 학습
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
model.fit(X_tr, y_tr)

# 3) 혼동행렬
pred = model.predict(X_te)
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

# 4) 정확도·정밀도·재현율·F1
acc = accuracy_score(y_te, pred)
prec = precision_score(y_te, pred)
rec = recall_score(y_te, pred)
f1 = f1_score(y_te, pred)
print(f"정확도={acc:.4f}, 정밀도={prec:.4f}, 재현율={rec:.4f}, F1={f1:.4f}")

# 5) 놓친 구매 세션 설명
n_actual_positive = y_te.sum()
miss_rate = fn / n_actual_positive
print(f"실제 구매 세션 {n_actual_positive}건 중 {fn}건을 놓쳤다 "
      f"(놓침 비율 {miss_rate:.1%}, 즉 재현율 {rec:.1%}의 여집합).")

TN=2002, FP=82, FN=186, TP=196
정확도=0.8913, 정밀도=0.7050, 재현율=0.5131, F1=0.5939
실제 구매 세션 382건 중 186건을 놓쳤다 (놓침 비율 48.7%, 즉 재현율 51.3%의 여집합).


<details>
<summary>(클릭) 💡 힌트</summary>

- 분할은 지난 순서와 같아야 비교가 성립합니다 — `test_size=0.2, random_state=42, stratify=y`.
- `confusion_matrix(y_te, pred).ravel()`은 `(TN, FP, FN, TP)` 순서로 네 값을 한 번에 풀어줍니다.
- 놓친 구매 세션은 **FN**(실제 구매인데 아니라고 예측)입니다. 비율은 `FN / 실제 구매 세션 수`입니다.
- 실제 구매 세션 수는 `y_te.sum()`입니다.

</details>

> 🎯 **[C2] 확인하기**  
> **382건 중 196건입니다. 약 51.3%입니다.**
>
> 정확도 0.8913이 어떻게 만들어졌는지 보면 명확합니다. 맞힌 2,198건 중 **2,002건이 TN** — "안 살 사람을 안 산다고 맞힌 것"입니다. 전체의 84.5%가 원래 안 사는 사람이니, 다수 클래스를 잘 맞히는 것만으로도 정확도가 높게 나타날 수 있습니다.
>
> 마케팅팀의 구매 세션 포착 목적과 직접 연결되는 지표는 **재현율 0.5131** — 음성으로 분류한 실제 구매 세션 **186건**입니다. 지난 두 순서 동안 우리는 이 숫자를 **한 번도 본 적이 없습니다.**

# 문제 2. 지표 5종을 교차 검증으로 한 번에 잰다

이제 지표를 늘립니다. 그리고 지난 순서에서 배운 대로, 한 번이 아니라 **5겹으로** 재서 평균 ± 표준편차로 적습니다.

불균형 데이터에서는 **평균 정밀도(Average Precision, AP)** (`average_precision`)를 함께 확인합니다. AP는 PR 곡선을 요약하지만 사다리꼴 적분으로 계산한 PR-AUC와 항상 같은 값은 아닙니다. 비교 대상이 되는 **기준선**도 함께 계산합니다 — 무작위 점수에서 기대되는 정밀도는 **양성 비율**입니다.

```
[문제 2]
1) cross_validate로 다섯 지표를 한 번에 구합니다.
   accuracy · precision · recall · f1 · average_precision(AP)
2) 각각 평균 ± 표준편차로 출력합니다.
3) AP를 양성 비율과 비교하고, 참고로 ROC-AUC도 구해 함께 확인합니다.
4) 다섯 지표 중 표준편차가 가장 큰 것이 무엇인지, 왜 그런지 설명합니다.
```

> 🤔 **예상하기**  
> 정확도의 표준편차는 지난 순서에서 **0.0035**였습니다. **재현율**의 표준편차는 그보다 클까요, 작을까요? (힌트: 각 검증 겹의 재현율은 약 382건의 양성 세션만 보고 계산됩니다)

▶️ **코드 실행하기 · 코드 셀 3 [C3]**

In [3]:
# [C3] 문제 2. 지표 5종을 교차 검증으로 한 번에 잰다
# ⌨️ 문제 2 — cross_validate로 다섯 지표를 동시에
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score

# 여기에 코드를 작성하세요 (5겹 CV로 지표 5종 → 평균±표준편차 → 기준선 대비)
# 0) 교차검증 설정 (분할 방식 고정)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1) 다섯 지표 + 참고용 roc_auc까지 한 번에 (같은 fold에서 일관되게 비교하려고 함께 포함)
scoring = ["accuracy", "precision", "recall", "f1", "average_precision", "roc_auc"]

results = cross_validate(model, X, y, cv=cv, scoring=scoring)

# 2) 평균 ± 표준편차 출력
for metric in scoring:
    scores = results[f"test_{metric}"]
    print(f"{metric:18s}: {scores.mean():.4f} ± {scores.std():.4f}")

# 3) AP를 양성 비율과 비교
baseline = y.mean()
ap_mean = results["test_average_precision"].mean()
print(f"\n양성 비율(baseline) = {baseline:.4f}")
print(f"AP 평균 = {ap_mean:.4f}  (baseline 대비 {ap_mean / baseline:.1f}배)")

accuracy          : 0.8969 ± 0.0034
precision         : 0.7232 ± 0.0117
recall            : 0.5409 ± 0.0241
f1                : 0.6186 ± 0.0175
average_precision : 0.7248 ± 0.0191
roc_auc           : 0.9091 ± 0.0081

양성 비율(baseline) = 0.1547
AP 평균 = 0.7248  (baseline 대비 4.7배)


In [4]:
# Recall의 표준편차가 가장 크다.
# 이 데이터는 양성 비율이 낮은 불균형 데이터인데, 5-fold로 나누면 각 테스트 부분에 들어가는 실제 양성 샘플 수는 더 적어진다.
# 재현율의 분모는 실제 양성 샘플 수이기 때문에 표본 수가 작으면 fold 간 변동도 더 커진다.

<details>
<summary>(클릭) 💡 힌트</summary>

- `cross_validate(model, X, y, cv=cv, scoring=["accuracy", "precision", "recall", "f1", "average_precision"])`
- 결과는 `dict`입니다. 각 지표는 `결과["test_accuracy"]` 처럼 `test_` 접두사가 붙은 키에 들어 있습니다.
- `average_precision`은 AP를 계산합니다. PR-AUC와 목적은 유사하지만 적분 방식이 달라 값이 항상 같지는 않습니다.
- 무작위 점수에서 기대되는 정밀도는 `y.mean()`입니다 — 아무 정보 없이 무작위로 찍었을 때의 값입니다.
- ROC-AUC는 확률이 필요합니다: `roc_auc_score(y_te, proba)`.

</details>

> 🎯 **[C3] 확인하기**  
> **재현율의 표준편차가 0.0257로 가장 큽니다 — 정확도(0.0035)의 7배가 넘습니다.**
>
> 이유는 분모에 있습니다. 각 검증 겹에서 정확도는 약 2,466건 전체로 계산하지만, 재현율은 그중 약 382건인 양성 세션으로 계산합니다. 분모가 더 작으므로 겹별 구성 변화에 더 민감할 수 있습니다. 그래서 **소수 클래스 지표를 보고할 때는 표준편차를 함께** 적어야 합니다.
>
> 그리고 두 요약값을 확인합니다. **ROC-AUC는 0.9002이고 AP는 0.7248입니다.** 두 값은 축과 계산 방식이 달라 크기를 직접 비교하지 않습니다. ROC-AUC는 양성과 음성의 순위 구분을 요약하고, AP는 양성 예측의 정밀도와 재현율을 요약합니다. AP 0.7248은 양성 비율 0.1547보다 높지만, 채택 여부는 운영점의 정밀도·재현율과 비용을 함께 확인해 판단합니다.
>
> **어느 지표로 보고할 것인가.** 마케팅팀의 관심은 "실제 구매 세션을 얼마나 포착하는가"이므로 **재현율**과 **AP**가 주 지표이고, 쿠폰이 낭비되지 않는지를 보는 **정밀도**가 함께 갑니다. 정확도는 보조 지표로 제시하되 단독으로 결론을 내리지 않습니다.

# 문제 3. 임계값을 훑어 운영점을 정한다

여기까지의 숫자는 전부 **임계값 0.5**에서 나온 것입니다. 그런데 0.5는 그냥 기본값일 뿐, 쿠폰 용량이나 오류 비용을 반영해 선택한 값은 아닙니다. 마케팅팀의 제약은 **쿠폰 500장**입니다 — 그 제약을 임계값으로 번역합니다.

```
[문제 3]
1) 임계값을 0.10부터 0.70까지 훑으며 양성 예측 건수 · 정밀도 · 재현율 · F1을 표로 만듭니다.
2) F1이 가장 높은 임계값을 찾습니다.
3) "쿠폰 500장" 제약을 만족하는 임계값을 역산합니다.
   (확률 상위 500건만 고르려면 임계값이 얼마여야 하는가)
4) 그 운영점에 실제 구매 세션이 몇 건 포함되는지 계산해 마케팅팀에 보고합니다.
```

> 🤔 **예상하기**  
> 기본 임계값 0.5에서는 양성 예측이 **278건**뿐이었습니다(쿠폰 500장을 다 못 씁니다). 임계값을 낮춰 500장을 꽉 채우면, 포함되는 실제 구매 세션이 196건에서 몇 건까지 늘어날지 예상합니다.

> ⚠️ **주의하기 — 운영점 선택과 최종 평가 분리**  
> 이 실습은 한정된 데이터로 임계값 선택 과정을 연습하므로 홀드아웃 데이터에서 후보 운영점을 탐색합니다. 실제 업무에서는 검증 데이터 또는 교차 검증의 OOF 예측으로 임계값을 선택하고, 테스트 데이터는 최종 평가에 한 번만 사용합니다. 따라서 여기서 얻은 0.2695는 학습용 후보값이며 바로 배포할 최종값이 아닙니다.

▶️ **코드 실행하기 · 코드 셀 4 [C4]**

In [5]:
# [C4] 문제 3. 임계값을 훑어 운영점을 정한다
# ⌨️ 문제 3 — 용량 제약(쿠폰 500장)을 임계값으로 번역하기

# 여기에 코드를 작성하세요 (임계값 표 → F1 최적 → 상위 500건 역산 → 성과 보고)
# 0) 준비: model을 X_tr로 학습해 X_te에 대한 확률 얻기
model.fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]

# 1) 임계값 0.10~0.70을 훑으며 표 만들기
rows = []
for t in np.arange(0.10, 0.71, 0.05):
    pred_t = (proba >= t).astype(int)
    n_pos = pred_t.sum()
    prec = precision_score(y_te, pred_t, zero_division=0)
    rec = recall_score(y_te, pred_t, zero_division=0)
    f1 = f1_score(y_te, pred_t, zero_division=0)
    rows.append({
        "임계값": round(t, 2),
        "양성예측건수": n_pos,
        "정밀도": round(prec, 4),
        "재현율": round(rec, 4),
        "F1": round(f1, 4),
    })

threshold_table = pd.DataFrame(rows)
print(threshold_table.to_string(index=False))

# 2) F1이 가장 높은 임계값 찾기
best_row = threshold_table.loc[threshold_table["F1"].idxmax()]
print(f"\nF1 최댓값 임계값 = {best_row['임계값']:.2f} (F1={best_row['F1']:.4f})")

# 3) "쿠폰 500장" 제약을 만족하는 임계값 역산
N_COUPON = 500
sorted_proba = np.sort(proba)[::-1]      # 내림차순 정렬
coupon_threshold = sorted_proba[N_COUPON - 1]   # 500번째로 높은 확률값 (0-indexed라 -1)
print(f"\n상위 {N_COUPON}건을 고르는 임계값 = {coupon_threshold:.4f}")

# 4) 그 운영점에서 실제 구매 세션이 몇 건 포함되는지 (TP)
pred_coupon = (proba >= coupon_threshold).astype(int)
n_selected = pred_coupon.sum()   # 실제 선택된 건수 (동점 처리로 500 초과 가능)
tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_te, pred_coupon).ravel()

print(f"실제 선택된 건수 = {n_selected}건 (동점 포함)")
print(f"그중 실제 구매 세션(TP) = {tp_c}건")
print(f"운영점 정밀도 = {tp_c / n_selected:.4f}")
print(f"운영점 재현율 = {tp_c / y_te.sum():.4f}")

print(f"\n[마케팅팀 보고] 쿠폰 {n_selected}장을 임계값 {coupon_threshold:.4f} 기준으로 발급하면, "
      f"그중 실제로 구매할 세션은 {tp_c}건으로 예상되며, "
      f"이는 전체 실제 구매 세션 {int(y_te.sum())}건 중 {tp_c/y_te.sum():.1%}를 포착하는 수준이다.")

 임계값  양성예측건수    정밀도    재현율     F1
0.10     729 0.4595 0.8770 0.6031
0.15     595 0.5227 0.8141 0.6366
0.20     543 0.5506 0.7827 0.6465
0.25     506 0.5652 0.7487 0.6441
0.30     482 0.5747 0.7251 0.6412
0.35     449 0.5991 0.7042 0.6474
0.40     396 0.6338 0.6571 0.6452
0.45     337 0.6736 0.5942 0.6314
0.50     278 0.7050 0.5131 0.5939
0.55     216 0.7824 0.4424 0.5652
0.60     183 0.8251 0.3953 0.5345
0.65     135 0.8741 0.3089 0.4565
0.70     112 0.8750 0.2565 0.3968

F1 최댓값 임계값 = 0.35 (F1=0.6474)

상위 500건을 고르는 임계값 = 0.2683
실제 선택된 건수 = 500건 (동점 포함)
그중 실제 구매 세션(TP) = 284건
운영점 정밀도 = 0.5680
운영점 재현율 = 0.7435

[마케팅팀 보고] 쿠폰 500장을 임계값 0.2683 기준으로 발급하면, 그중 실제로 구매할 세션은 284건으로 예상되며, 이는 전체 실제 구매 세션 382건 중 74.3%를 포착하는 수준이다.


<details>
<summary>(클릭) 💡 힌트</summary>

- 임계값을 적용한 예측은 `(proba >= t).astype(int)`입니다. `model.predict()`는 0.5로 고정이라 쓸 수 없습니다.
- 양성 예측 건수는 그 배열의 `.sum()`입니다.
- 정밀도 계산에서 양성 예측이 0건이면 오류가 납니다 — `precision_score(..., zero_division=0)`을 사용합니다.
- 상위 500건의 경계값은 확률을 **내림차순 정렬**해 500번째 값을 확인합니다: `np.sort(proba)[::-1][499]`. 경계에서 같은 점수가 여러 건이면 양성 예측이 500건을 넘을 수 있으므로 실제 운영에서는 동점 처리 규칙도 정합니다.
- 포함된 실제 구매 세션 수는 "양성 예측이면서 실제 양성"인 건수입니다 — 곧 **TP**입니다.

</details>

> 🎯 **[C4] 확인하기**  
> **포함되는 실제 구매 세션이 196건에서 284건으로 88건 늘어납니다.** 모델을 바꾸지도, 다시 학습시키지도 않았습니다. **임계값이라는 손잡이 하나**를 돌렸을 뿐입니다.
>
> 표를 세로로 확인합니다. 임계값이 내려갈수록 재현율은 오르고(0.2565 → 0.8770) 정밀도는 내려갑니다(0.8750 → 0.4595). 이 데이터에서는 두 지표 사이에 상충 관계가 나타납니다. 운영점은 모델 점수뿐 아니라 **용량과 비용 조건으로 정합니다.**
>
> F1이 가장 높은 지점은 0.20입니다. 하지만 우리가 고른 것은 **0.2695**입니다. 이는 학습용 홀드아웃에서 계산한 후보값이며, F1 최적점이 아니라 **예산이 감당하는 지점**입니다. 실무에서도 검증 데이터에서 이러한 운영 제약을 반영해 임계값을 선택합니다.

# 문제 4. `class_weight` 보정, 채택할 것인가

임계값 말고 다른 손잡이도 있습니다. **`class_weight="balanced"`** 는 학습 단계에서 소수 클래스에 가중치를 줘, 모델이 양성을 더 적극적으로 예측하게 만듭니다. 같은 목적에 도달하는 다른 길입니다.

두 방법은 작동 단계가 다르므로 **같은 검증 절차로 비교한 뒤 결정합니다.**

```
[문제 4]
1) class_weight="balanced"를 준 같은 모델로 교차 검증 지표 5종을 다시 구합니다.
2) 문제 2의 기본 설정과 나란히 표로 비교합니다.
3) 홀드아웃 혼동행렬도 함께 구해 TP·FN이 어떻게 달라졌는지 확인합니다.
4) 채택할 것인지 결정하고, 그 이유를 적습니다.
   (임계값 조정으로도 같은 효과를 낼 수 있다는 점을 함께 고려합니다)
```

> 🤔 **예상하기**  
> 개념 노트북의 데이터에서는 `class_weight`가 재현율을 크게 올리고 정밀도를 낮췄으며, F1은 소폭 올랐습니다. 이 데이터에서는 변화 폭이 어떻게 나타날지 예상합니다.

▶️ **코드 실행하기 · 코드 셀 5 [C5]**

In [6]:
# [C5] 문제 4. `class_weight` 보정, 채택할 것인가
# ⌨️ 문제 4 — 보정 전후를 같은 방식으로 비교

# 여기에 코드를 작성하세요 (balanced CV 지표 → 기본과 비교표 → 혼동행렬 대조)
# 0) cv_report 함수 정의 
def cv_report(model, X, y, cv, scoring, label):
    results = cross_validate(model, X, y, cv=cv, scoring=scoring)
    report = {"설정": label}
    for metric in scoring:
        scores = results[f"test_{metric}"]
        report[metric] = f"{scores.mean():.4f} ± {scores.std():.4f}"
    return report

# 1) 기본 모델 vs balanced 모델 정의
model_base = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
model_balanced = RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                          random_state=42, class_weight="balanced")

scoring = ["accuracy", "precision", "recall", "f1", "average_precision"]

report_base = cv_report(model_base, X, y, cv, scoring, "기본")
report_balanced = cv_report(model_balanced, X, y, cv, scoring, "balanced")

# 2) 나란히 표로 비교
compare_df = pd.DataFrame([report_base, report_balanced])
print("[교차검증 지표 비교]")
print(compare_df.round(4).to_string(index=False))

# 3) 홀드아웃 혼동행렬 비교
model_balanced.fit(X_tr, y_tr)
pred_balanced = model_balanced.predict(X_te)
tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_te, pred_balanced).ravel()

print(f"\n[홀드아웃 혼동행렬 비교]")
print(f"기본 모델     : TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"balanced 모델 : TN={tn_b}, FP={fp_b}, FN={fn_b}, TP={tp_b}")
print(f"\nTP 변화: {tp} → {tp_b} ({tp_b - tp:+d})")
print(f"FN 변화: {fn} → {fn_b} ({fn_b - fn:+d})")

[교차검증 지표 비교]
      설정        accuracy       precision          recall              f1 average_precision
      기본 0.8969 ± 0.0034 0.7232 ± 0.0117 0.5409 ± 0.0241 0.6186 ± 0.0175   0.7248 ± 0.0191
balanced 0.8646 ± 0.0051 0.5415 ± 0.0109 0.8202 ± 0.0234 0.6522 ± 0.0125   0.7176 ± 0.0158

[홀드아웃 혼동행렬 비교]
기본 모델     : TN=2002, FP=82, FN=186, TP=196
balanced 모델 : TN=1835, FP=249, FN=78, TP=304

TP 변화: 196 → 304 (+108)
FN 변화: 186 → 78 (-108)


In [7]:
# class_weight="balanced"를 채택하지 않는다.
# AP가 0.7248 → 0.7175로 거의 동일(오히려 미세하게 하락)한 것으로 볼 때, balanced 옵션이 모델의 순위 판별 능력 자체를 개선하지는 못했다.
# 재현율이 상승(0.5409→0.8092)했지만, 이는 기본 모델에서 임계값을 낮추는 것만으로도 동일하게 얻을 수 있는 효과이기 때문에 
# 별도로 모델을 재학습 할 필요 없이 기본 모델 + 임계값 조정이 더 효과적이라고 생각한다.

<details>
<summary>(클릭) 💡 힌트</summary>

- 문제 2에서 만든 `cv_report()` 함수를 그대로 재사용하면 두 설정을 같은 형식으로 잴 수 있습니다.
- `RandomForestClassifier(..., class_weight="balanced")` 한 인자만 추가하면 됩니다. 나머지는 동일하게 둡니다 — 그래야 비교가 성립합니다.
- 두 결과 `dict`를 `pd.DataFrame([base, balanced])`로 묶으면 표가 됩니다.
- 혼동행렬은 문제 1과 같은 방식으로, `balanced` 모델을 `X_tr`에 학습시킨 뒤 구합니다.

</details>

> 🎯 **[C5] 확인하기**  
> 개념 노트북과 마찬가지로 `class_weight`가 재현율(0.5430 → 0.8097)뿐 아니라 **F1까지 올렸습니다**(0.6204 → 0.6571). 변화의 크기는 데이터와 모델에 따라 달라집니다.
>
> 그래서 "`class_weight`는 F1을 떨어뜨린다" 같은 규칙을 외우면 안 됩니다. **검증 데이터에서 지표와 운영 비용을 다시 측정하는 것**이 필요합니다.
>
> 그런데 F1이 올랐는데도 채택하지 않는 것이 합리적일 수 있습니다. **AP가 0.7248에서 0.7176로 소폭 낮아졌다는 사실**도 함께 확인합니다. 이 실험에서는 `class_weight`로 순위 요약 성능이 개선되지 않았습니다. 기본 모델의 임계값 조정으로 비슷한 재현율을 얻을 수 있지만, 두 방법이 항상 같은 예측을 만드는 것은 아닙니다. **지표 하나가 올랐다고 채택하는 것이 아니라, 무엇이 실제로 개선됐는지를 보고 결정합니다.**

# 문제 5. 모델 카드 v4 완성 — 오늘의 제출물

v4에는 **지표 선택 근거**와 **운영점**이 들어갑니다. 이제 모델 카드가 "성능 기록"을 넘어 **운영 문서**가 됩니다.

```
[문제 5]
1) 임계값 표를 파일로 남깁니다 — threshold_table_v4.csv
2) 아래 모델 카드 v4 템플릿의 빈칸을 채웁니다.
   '지표 선택 근거'와 '운영점 선택 근거'는 오늘의 핵심이니 숫자를 붙여 적습니다.
```

▶️ **코드 실행하기 · 코드 셀 6 [C6]**

In [8]:
# [C6] 문제 5. 모델 카드 v4 완성 — 오늘의 제출물
# ⌨️ 문제 5 — 임계값 표를 파일로 남기기

# 여기에 코드를 작성하세요 (thr_tbl을 CSV로 저장하고 다시 읽어 확인)
threshold_table.to_csv("threshold_table_v4.csv", index=False, encoding="utf-8-sig")
print("저장 완료: threshold_table_v4.csv")

저장 완료: threshold_table_v4.csv


**모델 카드 v4 템플릿** — 아래 빈칸을 채워 이 셀에 완성합니다 (셀을 더블클릭해 편집).

```markdown
## 모델 카드 v4 — 구매 전환 예측

- 데이터: UCI Online Shoppers (12,330 세션, 수치형 10개 열, 양성 15.5%)
- 문제 유형: 분류 (불균형)
- 검증 방식: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
- 모델: RandomForest(n_estimators=300, min_samples_leaf=20)
- **주 지표와 선택 근거: 재현율·정밀도·F1·AP를 함께 본다. 양성 비율이 15.5%로 불균형하여, 정확도는 다수 클래스(미구매)만 맞혀도 84.5% 근처가 나올 수 있어 모델 성능을 확신하기 어렵다. 특히, 놓친 구매 세션(FN)이 실질적 손해로 이어지므로 재현율을 함께 봐야 한다.**
- **성능(CV, 평균 ± 표준편차)**
  - 정확도 { 0.8969 ± 0.0034 } / 정밀도 { 0.7232 ± 0.0117 } / 재현율 { 0.5409 ± 0.0241 } / F1 { 0.6186 ± 0.0175 } / AP { 0.7248 ± 0.0191 }
  - **양성 비율 { 0.1547 }와 AP { 0.7248 } 비교 (baseline 대비 4.7배)**
- **혼동행렬(임계값 0.5): TN { 2002 } / FP { 82 } / FN { 186 } / TP { 196 }**
  - **놓친 구매 세션 { 186 }건 ({ 48.7% }%)  ← 정확도만 봤을 때 보이지 않던 숫자**
- **운영점: 임계값 { 0.2683 } (용량 제약 { 500 }건)**
  - **그 지점의 정밀도 { 0.5680 } / 재현율 { 0.7435 } → 실제 구매 세션 { 284 }건 포함**
  - **선택 근거: 쿠폰 물량이 고정된 자원(예산 제약)이므로, F1을 최대화하는 임계값이 아니라 상위 500건을 정확히 골라내는 임계값을 역산해 운영점으로 채택했다.**
- **class_weight 보정: 채택 안 함 — 근거: AP가 0.7248 → 0.7175로 거의 변화가 없는 것을 볼 때, 모델의 순위 판별 능력 자체는 개선되지 않았다. 재현율 상승(0.5409→0.8092)과 정밀도 하락(0.7232→0.5529)은 판단 기준이 양성 쪽으로 옮겨간 결과로, 기본 모델에서 임계값을 낮추는 것만으로 동일한 효과를 재현할 수 있다. 재학습 없이 배포 후에도 유연하게 조정 가능한 임계값 조정이 더 실용적이다.**
- 한계 & 다음 단계:  
  1. 이 모델이 예측하는 것은 "구매 가능성이 높은 세션"이지, "쿠폰이 있어야만 사는 세션"이 아니다. 개입의 인과 효과를 봐야한다. 
  2. 원본 데이터의 범주형 8개 열(방문 요일, 브라우저, 지역 등)을 사용하지 않아 추가 개선 여지가 있다.  
  3. 임계값을 이번엔 홀드아웃에서 탐색했으나, 실제 배포 전에는 검증 데이터 또는 교차검증의 OOF 예측으로 다시 선택해야 한다.
- AI 사용 내역  
  무엇을 물었나: 혼동행렬·정밀도·재현율·F1·AP·ROC-AUC 계산 코드, class_weight 비교를 위한 cross_validate 코드, 임계값 탐색 
              및 상위 500건 역산 코드
  무엇을 검증했나: 각 지표의 정의와 계산 방식, zero_division 처리 등을 확인했고, class_weight 채택 여부는 AP 변화를 직접 비
                교해 스스로 판단함.
```

**제출:** 이 노트북(작성 셀 + 모델 카드 완성본)과 CSV를 개인 공개 저장소 main에 커밋·푸시하고, 저장소·커밋 링크를 제출합니다.

**스스로 점검하는 기준**

| 축 | 기준 |
| --- | --- |
| 지표 정합 | 주 지표가 문제 상황(용량 제약·불균형)에 맞는가 |
| 비용 논리 | 용량 제약을 임계값으로 정확히 번역했는가 |
| 수치 근거 | 주장에 실제 계산값이 붙어 있는가 (감이 아니라 숫자) |
| 결정의 정직성 | 보정을 채택/기각한 이유가 지표 변화로 설명되는가 |

> 🚀 **직접 확장하기**  
> 쿠폰 원가가 2,000원이고 추가 구매 1건의 이익이 20,000원이라면, **이익을 최대로 만드는 임계값**은 얼마일까요? 임계값 표에 `이익 = TP × 20000 − 양성예측 × 2000` 열을 추가해 계산합니다. 용량 제약이 없을 때 비용이 어떻게 운영점을 정하는지 직접 확인할 수 있습니다.

오늘 여러분은 **0.89 뒤에 가려져 있던 실제 구매 세션 186건**을 찾아냈습니다. 그리고 손잡이 하나를 돌려 실제 구매 세션 88건을 더 포함하는 후보 운영안을 만들었습니다.

숫자를 올리는 일보다, 그 숫자가 **누구를 세고 있는지** 묻는 일이 먼저입니다.

오늘도 한 걸음, 수고하셨습니다! 🎉
어제의 나보다 데이터를 다루는 손이 한 뼘 더 능숙해졌습니다. 다음 시간에 또 만나요.

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>